## Figure Reproduction for CEC Request

### Figure 1: Distribution of Daily Maximum Temperatures Averaged Across CA ISO Region 
Reproduction of a figure for CEC<br>
Notebook creation date: Aug 1, 2023<br> <br>
Modification from original figure: 
- show daily max instead of daily min 
- show just one season (summer) or month (july) -- hottest season/month instead of entire year 
- extend time period out to at least 2045 

#### Step 0: Import required libraries 
Here, we'll import the `climakitae` library, along with additional libraries used in the notebook.

In [ ]:
import climakitae as ck
import statsmodels.api as sm # Stats library for computing PDF 
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import calendar # Used for printing calendar names from integers (i.e 9 --> Sep)

# Plotting setting for in-notebook display 
%config InlineBackend.figure_format = 'retina'
mpl.rcParams['figure.dpi'] = 100 # Sets figure size in the notebook

#### Step 1: Read in daily maximum temperature 
We'll use the climakitae library to retrieve daily maximum WRF temperature from the cloud storage bucket.

In [ ]:
selections = ck.Select()

Next, we will make our data choices.

In [ ]:
# Set app.selections 
selections.timescale = "daily" 
selections.resolution = "9 km" 
selections.scenario_historical = ["Historical Climate"] 
selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"]
selections.data_type = "Gridded" 
selections.downscaling_method = ["Dynamical"] 
selections.time_slice = (1980, 2100)
selections.area_subset = "CA Electric Balancing Authority Areas" 
selections.cached_area = ["CALISO"] 
selections.variable = "Maximum air temperature at 2m" 
selections.units = "degF"
selections.area_average = "Yes"

Next, we'll retrieve the data from the catalog. The retrieval function will return an xarray Dataset object. 

In [ ]:
# Retrieve data
tmax_daily= selections.retrieve()
tmax_daily = tmax_daily.squeeze() # Remove singleton dimensions

#### Step 2: Select the time period of interest for the plot 
Choose to a month or set of months to subset the data by. <br><br>
For example, you may chose to generate the PDF and plot for just September, the 9th month of the year, using code in the following format: <br>
`data.isel(time = data.time.dt.month.isin([9]))`
<br><br>Or, you may want to look a the summer months-- June, July, and August (months 6, 7, & 8): <br> 
`data.isel(time = data.time.dt.month.isin([6,7,8]))`
<br><br>You could customize this subset of months however you want. For example, for California you may choose to extend the summer months through September, since September generally has high temperatures. 

In [ ]:
# Chose the months to subset by. Must be a list of integers, even if just looking at one month 
# month_subset = [6,7,8] # June, July, August
month_subset = [9] # September 

# Subset the data 
tmax_daily = tmax_daily.isel(time = tmax_daily.time.dt.month.isin(month_subset))

Because we're dealing with multiple time periods of interest and multiple simulations, it is more efficient to load all of `tmax_daily` before any subsetting. We'll do this now. 

In [ ]:
tmax_daily = ck.load(tmax_daily)

#### Step 3: Select your simulation of interest 
First, we'll print out the simulation options. 

In [ ]:
all_sims = tmax_daily.simulation.values
all_sims

Next, chose **one** simulation from the list of options

In [ ]:
sim_tmax_daily = tmax_daily.sel(simulation = all_sims[1])
print(all_sims[1])

#### Step 4: Split the data into historical and future time periods 
Next, we'll split the data into a historical period and a future period, by subsetting the time dimension. 
<br> You can easily change the ranges for these periods by modifying the code below. 

In [ ]:
# Choose the time periods of interest 
historical_period = ["1993","2007"]
current_period = ["2008","2022"]
nearfuture_period = ["2023","2037"]
midcentury_period = ["2038","2052"]

In [ ]:
# Split data into historical and future period 
tmax_historical = sim_tmax_daily.sel(time=slice(historical_period[0], historical_period[1]))
tmax_current = sim_tmax_daily.sel(time=slice(current_period[0], current_period[1]))
tmax_nearfuture = sim_tmax_daily.sel(time=slice(nearfuture_period[0], nearfuture_period[1]))
tmax_midcentury = sim_tmax_daily.sel(time=slice(midcentury_period[0], midcentury_period[1]))

As a last step, we'll read the data into memory to make the PDF and plotting steps faster. 

In [ ]:
# tmax_historical = ck.load(tmax_historical)
# tmax_future = ck.load(tmax_current)
# tmax_nearfuture = ck.load(tmax_nearfuture)
# tmax_midcentury = ck.load(tmax_midcentury)

#### Step 5: Compute the probability density function
First, we'll define a function that computes a probability density function. 

In [ ]:
def get_pdf(da): 
    """Compute probability density function 
    Applies statsmodels.api method to input DataArray 
    
    Args: 
        da (xr.DataArray): DataArray with only 1 dimension 
        
    Returns: 
        kde_da (xr.DataArray): PDF        
    """
    var_name = da.name # Use name of DataArray as dimension name 
    da = da.squeeze() # Squeeze out singleton dimensions 
    kde = sm.nonparametric.KDEUnivariate(da) # Compute PDF 
    kde.fit()
    kde_da = xr.DataArray(
        data = kde.density, 
        coords=[kde.support], 
        dims=[var_name], 
        name="Density"
    )
    return kde_da

Next, we'll apply that function to our data to generate the PDF.

In [ ]:
tmax_historical_pdf = get_pdf(tmax_historical) 
tmax_current_pdf = get_pdf(tmax_current) 
tmax_nearfuture_pdf = get_pdf(tmax_nearfuture)
tmax_midcentury_pdf = get_pdf(tmax_midcentury)

#### Step 6: Create the plot 
Finally, we'll plot up the data using matplotlib.

In [ ]:
# Create figure and axis objects, and add PDF lines to it 
fig, axs = plt.subplots(1)
pl_historical = tmax_historical_pdf.plot(
    ax=axs, 
    label="({0})".format(", ".join(historical_period)), 
    color="blue"
)
pl_current = tmax_current_pdf.plot(
    ax=axs, 
    label="({0})".format(", ".join(current_period)), 
    color="green"
)
pl_nearfuture = tmax_nearfuture_pdf.plot(
    ax=axs,
    label="({0})".format(", ".join(nearfuture_period)),
    color="orange"
)
pl_midcentury = tmax_midcentury_pdf.plot(
    ax=axs,
    label="({0})".format(", ".join(midcentury_period)),
    color="red"
)

# Add plot decorators (title, legend) 
axs.legend(title="Period") 
plt.suptitle("Distribution of Daily Maximum Temperatures Averaged Across CA ISO Region")
plt.title("{0}, Time period: {1}".format(all_sims[1], ", ".join([calendar.month_abbr[mon_index] for mon_index in month_subset])))
plt.legend(loc='upper left')

# Display the figure in the notebook
plt.show()

The above figure is just for a single simulation. Let's now repeat this for the remaining simulations. 

In [ ]:
for sim_name in all_sims:
    sim_tmax_daily = tmax_daily.sel(simulation = sim_name)
    
    tmax_historical = sim_tmax_daily.sel(time=slice(historical_period[0], historical_period[1]))
    tmax_current = sim_tmax_daily.sel(time=slice(current_period[0], current_period[1]))
    tmax_nearfuture = sim_tmax_daily.sel(time=slice(nearfuture_period[0], nearfuture_period[1]))
    tmax_midcentury = sim_tmax_daily.sel(time=slice(midcentury_period[0], midcentury_period[1]))
    
    tmax_historical_pdf = get_pdf(tmax_historical) 
    tmax_current_pdf = get_pdf(tmax_current) 
    tmax_nearfuture_pdf = get_pdf(tmax_nearfuture)
    tmax_midcentury_pdf = get_pdf(tmax_midcentury)

    
    # all_sims_ds = xr.concat([tmax_historical_pdf, tmax_current_pdf, tmax_nearfuture_pdf, tmax_midcentury_pdf], dim='simulation')
    
    # Create figure and axis objects, and add PDF lines to it 
    fig, axs = plt.subplots(1)
    pl_historical = tmax_historical_pdf.plot(
        ax=axs, 
        label="({0})".format(", ".join(historical_period)), 
        color="blue",
        alpha=0.5
    )
    pl_current = tmax_current_pdf.plot(
        ax=axs, 
        label="({0})".format(", ".join(current_period)), 
        color="green",
        alpha=0.5
    )
    pl_nearfuture = tmax_nearfuture_pdf.plot(
        ax=axs,
        label="({0})".format(", ".join(nearfuture_period)),
        color="orange",
        alpha=0.5
    )
    pl_midcentury = tmax_midcentury_pdf.plot(
        ax=axs,
        label="({0})".format(", ".join(midcentury_period)),
        color="red",
        alpha=0.5
    )

    # Add plot decorators (title, legend) 
    axs.legend(title="Period") 
    plt.suptitle("Distribution of Daily Maximum Temperatures Averaged Across CA ISO Region")
    plt.title("{0}, Time period: {1}".format(sim_name, ", ".join([calendar.month_abbr[mon_index] for mon_index in month_subset])))
    plt.legend(loc='upper left')

    # Display the figure in the notebook
    plt.show()

Next, some code to export the figure:

In [ ]:
# THIS IS ALL SIMS TOGETHER -- PRETTY MESSY
# Format the months used so that it can be used in the fig title 
months_list = [calendar.month_abbr[mon_index] for mon_index in month_subset]
figname = "cec_fig1_maxtemp_distribution_{0}_allmodels.png".format("".join(months_list))

# Save the figure. Set a high dpi to improve figure quality
fig.savefig(figname, dpi=200)

---
### Figure 2: Longest Stretch of Consecutive Extreme Heat Days per Year 
Modification from original figure: 
- update with data from Cal-Adapt: Analytics Engine
- original figure uses LOCA data
    - we will start with WRF, try with LOCA2

In [ ]:
import climakitae as ck
import pandas as pd
import numpy as np
import xarray as xr

from xclim.indices import heat_wave_max_length

#### Step 1: Select maximum and minimum air temperature data

This notebook makes use of the `heat_wave_max_length` function from the `xclim` package, and requires both maximum and minimum daily air temperature as inputs. We'll retrieve data from 1980-2100 for a location of interest under SSP 3-7.0.

In [ ]:
selections = ck.Select()
# selections.show()

In [ ]:
selections.downscaling_method=['Dynamical']
selections.timescale='daily'
selections.area_subset='CA counties'
selections.cached_area=['Sacramento County']
selections.time_slice=(1981,2100)
selections.scenario_historical=['Historical Climate']
selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
selections.area_average='Yes'
selections.units='degF'

selections.latitude=(38.59375-0.2, 38.59375+0.2)
selections.longitude=(-121.46875-0.2, -121.46875+0.2)

In [ ]:
selections.variable='Minimum air temperature at 2m'
tasmin = selections.retrieve()
tasmin = ck.load(tasmin)

In [ ]:
selections.variable='Maximum air temperature at 2m'
tasmax = selections.retrieve()
tasmax = ck.load(tasmax)

#### Step 2: Calculate the 98th percentile of historical daily maximum temperatures

We'll focus on the summer period of April through October, and use a baseline of 1961-1990 to assess the historical 98th percentile of daily maximum temperatures

In [ ]:
hist_tmax = tasmax.sel(time=slice('1981-04-01', '2010-12-31')) # WRF
# hist_tmax = tasmax.sel(time=slice('1961-04-01', '1990-10-31')) # LOCA

In [ ]:
# grab period between April and October
month_subset = [4, 5, 6, 7, 8, 9, 10]
hist_tasmax = hist_tmax.isel(time = hist_tmax.time.dt.month.isin(month_subset))

# calculate the 98th percentile
p98 = hist_tasmax.quantile(0.98).round(2).values
p98

#### Step 3: Calculate the longest stretch of consecutive extreme heat days per year

In [ ]:
ds = heat_wave_max_length(tasmin=tasmin, tasmax=tasmax,
                     window=1,
                     freq="YS", 
                     # thresh_tasmin = "{} degF".format(p98),
                     thresh_tasmax = "{} degF".format(p98),
                     op=">=")

In [ ]:
# plot
ds.hvplot.scatter(x='time', by='simulation', width=1000, height=400, alpha=1, ylabel='Max Consecutive Extreme Heat Duration (days)', title='Longest Stretch of Consecutive Extreme Heat Days per Year')

We'll also include some useful statistics of periods of interest, focusing on the historical period, mid-century (2035-2064), and end of century (2070-2099). 

In [ ]:
# HISTORICAL BASELINE (1981-2010)
ds_stats = ds.sel(time=slice('1981-01-01', '2010-12-31'))
print('30 historical year avg: {} days/yr'.format(ds_stats.mean()))
print('30 historical year range: {}-{} days/yr'.format(ds_stats.min(), ds_stats.max()))

# MID-CENTURY (2035-2064)
ds_stats = ds.sel(time=slice('2035-01-01', '2064-12-31'))
print('30 mid-century year avg: {} days/yr'.format(ds_stats.mean()))
print('30 mid-century year range: {}-{} days/yr'.format(ds_stats.min(), ds_stats.max()))

# END-CENTURY (2070-2099)
ds_stats = ds.sel(time=slice('2070-01-01', '2099-12-31'))
print('30 end of century year avg: {} days/yr'.format(ds_stats.mean()))
print('30 end of century year range: {}-{} days/yr'.format(ds_stats.min(), ds_stats.max()))

---
Development code below, for future modifications as requested

In [ ]:
#### Computing trendlines for each simulation
# trendlines = []

# for sim in ds.simulation:
#     one_sim = ds.sel(simulation=sim)
#     one_sim['time'] = one_sim.time.dt.year # Changing 'time' attribute to int for polyfit
#     m, b = one_sim.polyfit(dim='time', deg=1).polyfit_coefficients.values
#     trendline = m * one_sim.time + b
#     trendline['time'] = pd.to_datetime(trendline.time, format='%Y') # Converting years back to datetimes in trendline object
#     trendlines.append(trendline)
    
# # Combining trendlines across simulations
# trendlines = xr.concat(trendlines, dim='simulation')
# trendlines.name = 'Trendlines'
# tl = trendlines.hvplot.line(x='time', by='simulation')